## Imports and load

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

df = pd.read_csv("E:/project/Revolut-analysis-deashboard/Data/revolut_reviews_extended_analyzed.csv")
print(f"Loaded {len(df)} rows")

Loaded 282403 rows


## Feature engineering

In [2]:
df["is_one_star"] = (df["rating"] == 1).astype(int)
df["has_account_freeze"] = df["themes"].str.contains("account_freeze").astype(int)
df["has_customer_support"] = df["themes"].str.contains("customer_support").astype(int)
df["has_verification"] = df["themes"].str.contains("verification").astype(int)
df["has_fees"] = df["themes"].str.contains("fees").astype(int)
df["has_app_stability"] = df["themes"].str.contains("app_stability").astype(int)
df["has_transfers"] = df["themes"].str.contains("transfers").astype(int)

features = ["sentiment_score", "review_length", "has_account_freeze", "has_customer_support", 
            "has_verification", "has_fees", "has_app_stability", "has_transfers"]
X = df[features].fillna(0)
y = df["is_one_star"]

print(f"1-star reviews: {y.sum()} out of {len(y)} ({y.mean():.1%})")

1-star reviews: 34140 out of 282403 (12.1%)


## Train the model

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.92      0.94     49653
           1       0.54      0.69      0.60      6828

    accuracy                           0.89     56481
   macro avg       0.75      0.80      0.77     56481
weighted avg       0.91      0.89      0.90     56481



## Predictive Model: What Drives a 1-Star Review?

**Method:** Trained a Random Forest Classifier to predict whether a review is 1-star, 
using 8 features: sentiment score, review length, and whether the review mentions each of 
6 complaint themes (account freeze, customer support, verification, fees, app stability, 
transfers). Class weighting was applied to account for the imbalance between 1-star and 
non-1-star reviews.

### Model Performance

| Class | Precision | Recall | F1-score | Support (actual count) |
|---|---|---|---|---|
| Not 1-star (0) | 0.96 | 0.92 | 0.94 | 49,653 |
| 1-star (1) | 0.54 | 0.69 | 0.60 | 6,828 |
| **Overall accuracy** | | | **0.89** | 56,481 |

**What this means:**
- The model correctly identifies **69% of all genuine 1-star reviews** (recall)
- When the model flags a review as 1-star, it's correct **54% of the time** (precision)
- 1-star reviews make up only ~12% of the dataset, making them a harder minority class to 
  predict — the weaker scores on this class reflect that difficulty, not a flawed model

**Honest assessment:** this is a moderate, usable model — not highly precise, but a 
legitimate working classifier that supports the more important output below: feature 
importance.

### Feature Importance (the key output)

[Insert Cell 4 output here once run — ranked list of which features mattered most]

### Why this matters for the project

This moves the analysis from *descriptive* ("here's what we observed in the data") to 
*predictive* ("here's a trained model that quantifies which factors most strongly drive a 
1-star review"). The feature importance ranking provides a data-backed answer to which 
specific complaint themes deserve the highest priority in business recommendations, rather 
than relying on raw frequency counts alone.

**Limitation:** the model uses only theme flags and basic text metrics as features — it 
does not use the full text content directly (e.g., via NLP embeddings), which could 
improve predictive performance further in future iterations.

## Feature importance

In [4]:
importance_df = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance_df)

                feature  importance
0       sentiment_score    0.561145
1         review_length    0.345359
3  has_customer_support    0.035486
2    has_account_freeze    0.019997
4      has_verification    0.016055
6     has_app_stability    0.013484
5              has_fees    0.004592
7         has_transfers    0.003882


## Save

In [5]:
importance_df.to_csv("E:/project/Revolut-analysis-deashboard/Data/revolut_feature_importance.csv", index=False)
print("Saved.")

Saved.
